# Базовый эксперимент с SIR моделью

**Цель:** Запустить агентную SIR модель с параметрами по умолчанию
и проанализировать динамику эпидемии.

## 1. Инициализация проекта

Подключаем необходимые пакеты и активируем окружение.

In [1]:
using DrWatson
@quickactivate

using Agents, DataFrames, Plots
using JLD2

Подключаем модель

In [2]:
include(srcdir("sir_model.jl"))

total_count (generic function with 1 method)

## 2. Параметры эксперимента

Определяем параметры модели:
- **Ns** — численность населения в трёх городах (по 1000 человек)
- **β_und** — вероятность заражения невыявленными (0.5)
- **β_det** — вероятность заражения выявленными (0.05)
- **infection_period** — длительность болезни (14 дней)
- **detection_time** — время до выявления (7 дней)
- **death_rate** — вероятность смерти (2%)
- **reinfection_probability** — вероятность повторного заражения (10%)
- **Is** — начальное количество инфицированных (только в городе 3)
- **seed** — зерно случайности

In [3]:
params = Dict(
    :Ns => [1000, 1000, 1000],
    :β_und => [0.5, 0.5, 0.5],
    :β_det => [0.05, 0.05, 0.05],
    :infection_period => 14,
    :detection_time => 7,
    :death_rate => 0.02,
    :reinfection_probability => 0.1,
    :Is => [0, 0, 1],
    :seed => 42,
)

Dict{Symbol, Any} with 9 entries:
  :Is                      => [0, 0, 1]
  :seed                    => 42
  :infection_period        => 14
  :β_und                   => [0.5, 0.5, 0.5]
  :Ns                      => [1000, 1000, 1000]
  :detection_time          => 7
  :reinfection_probability => 0.1
  :β_det                   => [0.05, 0.05, 0.05]
  :death_rate              => 0.02

## 3. Расчёт базового репродуктивного числа

Базовое репродуктивное число R₀ = β/γ, где γ = 1/infection_period

- β = 0.5
- γ = 1/14 ≈ 0.0714
- R₀ = 0.5 / 0.0714 ≈ 7.0

In [4]:
γ = 1 / params[:infection_period]
R₀ = params[:β_und][1] / γ
println("Базовое репродуктивное число R₀ = ", round(R₀, digits=2))

Базовое репродуктивное число R₀ = 7.0


## 4. Запуск симуляции

In [5]:
println("Инициализация модели...")
model = initialize_sir(; params...)

Инициализация модели...


StandardABM with 3000 agents of type Person
 agents container: Dict
 space: GraphSpace with 3 positions and 3 edges
 scheduler: fastest
 properties: C, infection_period, β_und, Ns, migration_rates, detection_time, rng, reinfection_probability, β_det, death_rate

Массивы для хранения данных

In [6]:
times = Int[]
S_vals = Int[]
I_vals = Int[]
R_vals = Int[]
total_vals = Int[]

n_steps = 100
println("Запуск симуляции на $n_steps шагов...")

for step = 1:n_steps
    Agents.step!(model, 1)

    push!(times, step)
    push!(S_vals, susceptible_count(model))
    push!(I_vals, infected_count(model))
    push!(R_vals, recovered_count(model))
    push!(total_vals, total_count(model))

    if step % 10 == 0
        println("  Шаг $step: S=$(S_vals[end]), I=$(I_vals[end]), R=$(R_vals[end])")
    end
end

Запуск симуляции на 100 шагов...
  Шаг 10: S=2647, I=353, R=0
  Шаг 20: S=0, I=2989, R=8
  Шаг 30: S=0, I=2951, R=1
  Шаг 40: S=0, I=2681, R=229
  Шаг 50: S=0, I=2835, R=51
  Шаг 60: S=0, I=2818, R=14
  Шаг 70: S=0, I=2781, R=2
  Шаг 80: S=0, I=2724, R=0
  Шаг 90: S=0, I=2598, R=102
  Шаг 100: S=0, I=2640, R=25


## 5. Сохранение данных

In [7]:
agent_df = DataFrame(time = times, susceptible = S_vals, infected = I_vals, recovered = R_vals)
model_df = DataFrame(time = times, total = total_vals)

println("\nСохранение данных...")
@save datadir("sir_basic_agent.jld2") agent_df
@save datadir("sir_basic_model.jld2") model_df


Сохранение данных...


## 6. Визуализация результатов

In [8]:
println("Построение графика...")
plot(agent_df.time, agent_df.susceptible,
     label = "Восприимчивые (S)",
     xlabel = "Дни",
     ylabel = "Количество",
     linewidth = 2,
     color = :blue)
plot!(agent_df.time, agent_df.infected,
      label = "Инфицированные (I)",
      linewidth = 2,
      color = :red)
plot!(agent_df.time, agent_df.recovered,
      label = "Выздоровевшие (R)",
      linewidth = 2,
      color = :green)
plot!(agent_df.time, model_df.total,
      label = "Всего (включая умерших)",
      linestyle = :dash,
      linewidth = 2,
      color = :black)

title!("Модель SIR: Динамика эпидемии (R₀ = $(round(R₀, digits=2)))")
savefig(plotsdir("sir_basic_dynamics.png"))

Построение графика...


"/afs/.dk.sci.pfu.edu.ru/home/m/v/mvchuvakina/work/study/2026-1/2026-1--study--simulation-modeling/labs/lab04/project/plots/sir_basic_dynamics.png"

## 7. Анализ результатов

In [9]:
println("\n=== Анализ результатов ===")
println("Пик заболеваемости: I_max = ", maximum(I_vals))
println("Время достижения пика: ", argmax(I_vals), " дней")
println("Итоговое число переболевших: R(∞) = ", R_vals[end])
println("Доля переболевших: ", round(R_vals[end]/3000*100, digits=1), "%")
println("Всего умерших: ", 3000 - total_vals[end])


=== Анализ результатов ===
Пик заболеваемости: I_max = 3000
Время достижения пика: 15 дней
Итоговое число переболевших: R(∞) = 25
Доля переболевших: 0.8%
Всего умерших: 335


**Выводы:**
- При R₀ = $(round(R₀, digits=2)) эпидемия развивается очень быстро
- Пик достигается на $(argmax(I_vals))-й день
- Высокая заразность приводит к тому, что почти все восприимчивые заболевают